# 1. Lists, tuples, sets and dictionaries

Learn to choose a container based on ordering, repetition, mutability, and lookup needs. You will preserve paired records, count repeated categories, and explain why a set cannot replace a list of observations. Prerequisite: [Strings](03_strings.ipynb). Allow two hours, including drawing the contents of each container before inspecting Python's output. No machine learning model is fitted in this lesson.

## 2. What problem does this solve?

A single variable cannot conveniently represent an entire dataset. We need collections: an ordered sequence of readings, a fixed pair of coordinates, a collection of distinct labels, or a mapping from a product name to its price. Python provides different containers because these questions have different requirements.

The distinction affects correctness. Removing repeated values changes observed frequencies. Reordering only one of two paired columns changes which outcome belongs to which example. Choosing a container is therefore a decision about data meaning, not only convenient notation.

## 3. Intuition and analogy

A list resembles an ordered shopping receipt: the same product can appear twice and position is recorded. A tuple resembles a named arrangement of slots that should not be rearranged, such as a coordinate pair. A set resembles a guest checklist containing each distinct name once. A dictionary resembles a lookup directory: a key leads to its associated value.

No analogy captures every rule. A tuple can contain a mutable list, so fixing its slots does not make every nested object immutable. A dictionary preserves insertion order, but its main relationship is key to value. Sorting entries is a separate decision that must preserve the correct pairing.

## 4. Terminology

A **list** uses brackets, such as `[2, 1, 3]`, retains order and duplicates, and allows changes. A **tuple** uses commas, often inside parentheses, such as `(20, 30)`, and cannot have its entries replaced. A one-element tuple needs a comma: `(20,)`.

A **set** contains distinct hashable values, such as `{'A', 'B'}`. Do not rely on its iteration order. An empty set is `set()`, whereas `{}` creates an empty dictionary. A **dictionary** maps unique hashable keys to values, such as `{'A': 10}`. Hashability is a requirement for stable lookup; strings and numbers are common keys, while mutable lists cannot be keys.

**Membership** asks whether a value is present using `in`. **Aliasing** means multiple names refer to one object. A **shallow copy** creates a new outer container while potentially sharing nested objects; a **deep copy** recursively copies supported nested contents.

## 5. Mathematical foundations

Consider product labels $p_i$ and quantities $q_i$ for n sale lines, where i identifies a row. Total quantity for product g is

$$Q_g=\sum_{i=1}^{n}\mathbf{1}[p_i=g]q_i.$$

The indicator $\mathbf{1}[p_i=g]$ is one for matching products and zero otherwise. Quantities have units of items; total $Q_g$ also has units of items. The product-label list and quantity list must both have length n, and their positions must correspond to the same sale line.

The set of product labels answers a different question: which products appeared? Its size counts distinct categories, not sale lines. Discarding repeated labels loses the information required to count observations faithfully.

## 6. Hand-calculated example

There are three sale lines: A with quantity two, B with quantity one, and A again with quantity three. The ordered labels are `[A,B,A]`. There are three observations but only two distinct products. Product totals are $Q_A=2+3=5$ items and $Q_B=1$ item; all quantities sum to six.

If we convert the labels to a set and then pair its two members with the original three quantities, the lengths and meanings no longer match. If we use a dictionary literal with A twice, the later A value replaces the earlier one instead of automatically adding quantities.

## 7. Implementation from scratch

The short loop below visits paired labels and quantities once each. `zip` pairs entries by position; the length check ensures it does not silently discard an unmatched tail. The next lesson explains iteration and conditions more fully.

In [ ]:
# Enable embedded figures in this fresh notebook kernel.
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

In [ ]:
products = ['A', 'B', 'A']
quantities = [2, 1, 3]
assert len(products) == len(quantities)
totals = {}
for product, quantity in zip(products, quantities):
    totals[product] = totals.get(product, 0) + quantity
print('Product totals:', totals)
assert totals == {'A': 5, 'B': 1}
assert sum(totals.values()) == sum(quantities) == 6

## 8. Step-by-step output inspection

In [ ]:
coordinate = (20.0, 30.0)
distinct_products = set(products)
prices = {'A': 10, 'B': 20}
print('Observations:', len(products), 'Distinct products:', len(distinct_products))
print('Coordinate first component:', coordinate[0])
print('Price for A:', prices['A'])
assert distinct_products == {'A', 'B'}
assert len(products) == 3 and len(distinct_products) == 2
overwritten = {'A': 2, 'B': 1, 'A': 3}
assert overwritten['A'] == 3

The repeated key demonstration intentionally shows replacement. It is useful when updating a known value, but wrong when the intended operation is accumulation. Dictionary keys are unique even when the raw observations contain repeated categories.

## 9. Standard-library implementation

`Counter` records occurrence counts, not quantities sold. `defaultdict` can supply a starting zero for a numerical accumulation. Comparing them clarifies two different meanings of a count.

In [ ]:
from collections import Counter, defaultdict
line_counts = Counter(products)
quantity_totals = defaultdict(int)
for product, quantity in zip(products, quantities):
    quantity_totals[product] += quantity
assert dict(line_counts) == {'A': 2, 'B': 1}
assert dict(quantity_totals) == totals
print('Sale-line counts:', dict(line_counts))
print('Item quantities:', dict(quantity_totals))

Two lines for A contain five items. Neither result is revenue; calculating money would additionally require a valid price for each line. A correct container cannot supply information that the dataset does not contain.

## 10. Visualization

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
ordered_products = sorted(totals)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].bar(ordered_products, [line_counts[key] for key in ordered_products])
axes[0].set(title='Recorded sale lines', xlabel='Product', ylabel='Number of lines', ylim=(0, 3))
axes[1].bar(ordered_products, [totals[key] for key in ordered_products])
axes[1].set(title='Items sold', xlabel='Product', ylabel='Quantity (items)', ylim=(0, 6))
fig.tight_layout()
display(fig)
plt.close(fig)

The panels use the same products but different measured quantities and vertical scales. The left shows A appears twice; the right shows A contributes five items. Reading both as generic “counts” would obscure the distinction we need for a correct report.

## 11. Realistic experiment: preserving raw records

Storing one dictionary per row keeps related fields together. Copying only the outer list still shares its row dictionaries. Use a deliberate independent copy before changing nested teaching data that should leave the original intact.

In [ ]:
from copy import deepcopy
raw_records = [{'sale_id': 101, 'product': 'A', 'quantity': 2},
               {'sale_id': 102, 'product': 'B', 'quantity': 1}]
shallow = raw_records.copy()
assert shallow is not raw_records and shallow[0] is raw_records[0]
working = deepcopy(raw_records)
working[0]['quantity'] = 99
assert raw_records[0]['quantity'] == 2
assert working[0]['quantity'] == 99
print('Raw first row:', raw_records[0], 'Independent working row:', working[0])

Deep copying is reasonable for this tiny nested collection. Large numerical workflows may instead use immutable raw files and explicitly created derived tables to avoid unnecessary memory costs. The important requirement is that a transformation's effect on its inputs is understood and documented.

## 12. Representation choices

Choose a list when order and repetitions matter, a tuple when an arrangement should stay fixed, a set when only distinct membership matters, and a dictionary when keys identify values. These are programming choices, not model hyperparameters. For an ML feature vector, preserve an explicit feature order; converting feature names to a set can break correspondence with learned coefficients.

## 13. Evaluation

Check observation count, distinct-key count, pairing, and reconciled totals. Test repeated categories and unknown keys rather than only unique happy-path examples. An absent dictionary key raises `KeyError` when indexed; `.get(key, default)` supplies a fallback. Use a fallback only when that value is justified: a missing price should not silently become zero.

## 14. Assumptions and limitations

The example assumes quantities are valid item counts and product labels have already been normalized correctly. It does not handle returns, malformed records, or changing prices. Average dictionary and set lookup is efficient for common use, but performance depends on implementation and keys. Choose structures for meaning first, then measure a real bottleneck before optimizing.

## 15. Common mistakes and debugging

Do not expect set order to define dataset order. Do not assume assigning a list creates a copy. Do not think a tuple makes nested lists immutable. Avoid modifying a collection while iterating over it unless the behavior is deliberately designed. Use `sorted(records, key=...)` on complete records rather than sorting feature and target lists independently.

## 16. Comparison with alternatives

Python containers are flexible and readable for small mixed collections. NumPy arrays provide homogeneous numerical shapes and vectorized operations. pandas DataFrames provide labeled columns and table operations. A list of dictionaries is a useful bridge to a DataFrame because each dictionary naturally represents one row, but its schema still needs validation.

## 17. Exercises

1. **Beginner:** Choose a container for ordered repeated readings, distinct product names, and a product-to-price lookup.
2. **Beginner:** For labels `[A,A,B,C,B]`, calculate observation count and distinct-product count.
3. **Beginner:** Explain the result of `{'A': 2, 'A': 7}` and how it differs from adding quantities.
4. **Intermediate:** Show why a shallow copy of a list of dictionaries does not protect the original row dictionaries.
5. **Intermediate:** Explain how independently sorting ages and outcomes can produce incorrect training examples.
6. **Challenge:** Accumulate quantities from row dictionaries, reject an unknown product, and reconcile totals against accepted input quantities without modifying the original rows.

## 18. Detailed solutions

1. Use a list, set, and dictionary respectively. A tuple is another option for a fixed sequence whose entries should not be rebound.
2. Five observations include three distinct products. A occurs twice, B twice, and C once.
3. The value is seven because the later occurrence replaces the earlier value. Accumulation would explicitly calculate two plus seven, giving nine.
4. The new outer list still refers to the original inner dictionaries. A deep copy or deliberately constructed independent rows avoid this shared mutation.
5. Sorting changes row pairing unless every associated field moves with its entire record. Equal lengths alone cannot detect corrupted correspondence.
6. The following solution uses a known product vocabulary, validates counts, and builds a separate result. These tiny records are synthetic.

In [ ]:
rows = [{'product': 'A', 'quantity': 2}, {'product': 'B', 'quantity': 1},
        {'product': 'A', 'quantity': 3}, {'product': 'UNKNOWN', 'quantity': 9}]
accepted_total = 0
rejected = []
answer = {}
for row in rows:
    product, quantity = row['product'], row['quantity']
    if product not in {'A', 'B'}:
        rejected.append(row.copy())
        continue
    if type(quantity) is not int or quantity < 0:
        raise ValueError('Expected a nonnegative integer item count')
    answer[product] = answer.get(product, 0) + quantity
    accepted_total += quantity
assert answer == {'A': 5, 'B': 1}
assert sum(answer.values()) == accepted_total == 6
assert len(rejected) == 1 and len(rows) == 4
print('Accepted totals:', answer, 'Rejected rows:', rejected)

## 19. Knowledge check

**Does a set preserve duplicates?** No. **Are dictionary keys unique?** Yes; assigning an existing key updates its value. **Is a tuple recursively immutable?** No; it can contain mutable objects. **Why preserve row pairing?** Each feature must still correspond to its actual outcome. **Does a shallow copy duplicate nested objects?** Usually not; it copies the outer container.

## 20. Interview preparation

**When choose a dictionary over a list?** When a key-to-value relationship supports the intended lookup. **Why is set conversion dangerous for observations?** It removes repetition and loses order needed for frequencies or pairing. **What is aliasing?** Several references point to one object. **Why validate keys before joins or lookups?** Missing or duplicated identity rules can change numerical meaning. **How preserve a dataset during transformation?** Keep raw inputs immutable by policy and create explicit independent derived objects when needed.

## 21. Summary and next steps

Containers encode different promises about data. Select one deliberately, and check that operations preserve the observation unit. Continue to [Conditions and loops](05_conditions_and_loops.ipynb) to learn the control flow used in the accumulation examples.